# Wildfire preprocessing — leak-free, chronological split

**Milestone 3 | From cleaned source data to numeric model-ready files**

This notebook prepares the California daily wildfire-start dataset for a later model-training milestone. It excludes records after 2023 because the full 2024 year and the partial 2025 segment have no positive labels, then sorts chronologically and holds out the final 20% of dates. The target is `FIRE_START_DAY` on the same date, so this is a historical-day classification dataset, not a claim of advance warning. Daily weather measurements may be contemporaneous with fire starts; a true operational early-warning study would require carefully lagged predictors and timestamp validation.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

_candidates=[Path.cwd(), *Path.cwd().parents]
ROOT=next(p for p in _candidates if (p/'data/raw/CA_Weather_Fire_Dataset_1984-2025.csv').exists())
df=pd.read_csv(ROOT/'data/raw/CA_Weather_Fire_Dataset_1984-2025.csv',parse_dates=['DATE'])
df=df.loc[df.YEAR<=2023].sort_values('DATE').reset_index(drop=True)
TARGET='FIRE_START_DAY'
NUMERIC=['PRECIPITATION','MAX_TEMP','MIN_TEMP','AVG_WIND_SPEED','YEAR','TEMP_RANGE','WIND_TEMP_RATIO','MONTH','LAGGED_PRECIPITATION','LAGGED_AVG_WIND_SPEED','DAY_OF_YEAR']
CATEGORICAL=['SEASON']
print('Rows:',len(df),'| period:',df.DATE.min().date(),'to',df.DATE.max().date(),'| features before encoding:',len(NUMERIC)+len(CATEGORICAL))
df[NUMERIC+CATEGORICAL+[TARGET]].head()

Rows: 14610 | period: 1984-01-01 to 2023-12-31 | features before encoding: 12


,PRECIPITATION,MAX_TEMP,MIN_TEMP,AVG_WIND_SPEED,YEAR,TEMP_RANGE,WIND_TEMP_RATIO,MONTH,LAGGED_PRECIPITATION,LAGGED_AVG_WIND_SPEED,DAY_OF_YEAR,SEASON,FIRE_START_DAY
0,0.0,79.0,51.0,4.70,1984,28.0,0.059494,1,0.0,4.700,1,Winter,False
1,0.0,71.0,46.0,5.59,1984,25.0,0.078732,1,0.0,5.145,2,Winter,False
2,0.0,70.0,47.0,5.37,1984,23.0,0.076714,1,0.0,5.220,3,Winter,False
3,0.0,76.0,45.0,4.70,1984,31.0,0.061842,1,0.0,5.090,4,Winter,False
4,0.0,74.0,49.0,5.14,1984,25.0,0.069459,1,0.0,5.100,5,Winter,False


## Feature decisions

`DATE` is used for ordering and leakage checks but is excluded from the input matrix. `FIRE_START_DAY` is the target and is never an input. The source weather variables are retained. `TEMP_RANGE`, `WIND_TEMP_RATIO`, and the lagged weather summaries are retained as documented engineered/source-derived features because they represent daily temperature spread, wind relative to temperature, and prior-week weather. `SEASON` is one-hot encoded. Calendar fields remain numeric, and numeric inputs are standardized. No feature is selected based on a model score at this milestone.

## Split first, then fit preprocessing on training data

The data are sorted by date. The first 80% form training data and the final 20% form the held-out test period. Numeric medians, standardization parameters, and categorical encoding are all learned from training rows only. This preserves time order and prevents information from the test period from influencing preprocessing.

In [2]:
cut=int(len(df)*0.80)
train=df.iloc[:cut].copy(); test=df.iloc[cut:].copy()
print('Train:',len(train),train.DATE.min().date(),'to',train.DATE.max().date())
print('Test :',len(test),test.DATE.min().date(),'to',test.DATE.max().date())
print('Target rates | train:',round(train[TARGET].mean(),3),'test:',round(test[TARGET].mean(),3))
assert train.DATE.max()<test.DATE.min()
assert set(train.DATE).isdisjoint(set(test.DATE))
try:
    encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError:
    encoder=OneHotEncoder(handle_unknown='ignore',sparse=False)
preprocessor=ColumnTransformer(transformers=[
    ('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),NUMERIC),
    ('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',encoder)]),CATEGORICAL)
],remainder='drop',verbose_feature_names_out=False)
preprocessor.fit(train[NUMERIC+CATEGORICAL])
X_train=pd.DataFrame(preprocessor.transform(train[NUMERIC+CATEGORICAL]),columns=preprocessor.get_feature_names_out())
X_test=pd.DataFrame(preprocessor.transform(test[NUMERIC+CATEGORICAL]),columns=preprocessor.get_feature_names_out())
y_train=train[TARGET].astype(int).reset_index(drop=True)
y_test=test[TARGET].astype(int).reset_index(drop=True)
display(X_train.head())
print('X_train:',X_train.shape,'X_test:',X_test.shape,'numeric columns:',X_train.shape[1])

Train: 11688 1984-01-01 to 2015-12-31
Test : 2922 2016-01-01 to 2023-12-31
Target rates | train: 0.31 test: 0.46


,PRECIPITATION,MAX_TEMP,MIN_TEMP,AVG_WIND_SPEED,YEAR,TEMP_RANGE,WIND_TEMP_RATIO,MONTH,LAGGED_PRECIPITATION,LAGGED_AVG_WIND_SPEED,DAY_OF_YEAR,SEASON_Fall,SEASON_Spring,SEASON_Summer,SEASON_Winter
0,-0.180068,1.195164,-0.792621,-1.298840,-1.678626,2.328774,-1.353869,-1.601451,-0.348969,-1.976221,-1.727310,0.0,0.0,0.0,1.0
1,-0.180068,0.085880,-1.534192,-0.878632,-1.678626,1.828449,-0.810804,-1.601451,-0.348969,-1.656621,-1.717826,0.0,0.0,0.0,1.0
2,-0.180068,-0.052781,-1.385878,-0.982503,-1.678626,1.494898,-0.867771,-1.601451,-0.348969,-1.602756,-1.708342,0.0,0.0,0.0,1.0
3,-0.180068,0.779182,-1.682507,-1.298840,-1.678626,2.829100,-1.287578,-1.601451,-0.348969,-1.696122,-1.698858,0.0,0.0,0.0,1.0
4,-0.180068,0.501861,-1.089249,-1.091097,-1.678626,1.828449,-1.072558,-1.601451,-0.348969,-1.688940,-1.689374,0.0,0.0,0.0,1.0


X_train: (11688, 15) X_test: (2922, 15) numeric columns: 15


## Final readiness checks and exports

All model inputs are numeric. The model-ready feature matrices contain no missing or infinite values. Scaling is applied to numeric columns only; the one-hot season columns remain 0/1 indicators. The target and date audit files are separate from `X_train` and `X_test`. The saved `preprocessing_pipeline.joblib` should be reused to transform future data; do not fit a new scaler on the test set.

In [3]:
out=ROOT/'data/model_ready'; out.mkdir(parents=True,exist_ok=True)
X_train.to_csv(out/'X_train.csv',index=False); X_test.to_csv(out/'X_test.csv',index=False)
y_train.rename(TARGET).to_csv(out/'y_train.csv',index=False); y_test.rename(TARGET).to_csv(out/'y_test.csv',index=False)
train[['DATE']].to_csv(out/'train_dates.csv',index=False); test[['DATE']].to_csv(out/'test_dates.csv',index=False)
joblib.dump(preprocessor,out/'preprocessing_pipeline.joblib')
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train.dtypes)
assert X_train.isna().sum().sum()==0 and X_test.isna().sum().sum()==0
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all()
assert not set(train.DATE).intersection(set(test.DATE))
print('Train/test overlap:',len(set(train.DATE).intersection(set(test.DATE))))
print('Missing cells | train:',int(X_train.isna().sum().sum()),'test:',int(X_test.isna().sum().sum()))
print('Train positives:',int(y_train.sum()),f'({y_train.mean():.1%})')
print('Test positives:',int(y_test.sum()),f'({y_test.mean():.1%})')
print('Saved X_train, X_test, y_train, y_test, date audits, and fitted training-only pipeline to',out)

Train/test overlap: 0
Missing cells | train: 0 test: 0
Train positives: 3627 (31.0%)
Test positives: 1344 (46.0%)
Saved X_train, X_test, y_train, y_test, date audits, and fitted training-only pipeline to /home/ubuntu/work/wildfire/wildfire_project/data/model_ready


## Final hand-off

The dataset is prepared for Module 4 model training, not trained here. Preserve the date-based split for all model comparisons. Because the test label rate differs from the training rate, report this temporal distribution shift and compare results with appropriate metrics in the next milestone. The same-day target and same-day weather features must not be described as a real-time early-warning system without a stricter prediction-time design.